# Reconocimiento de Lenguaje de Señas (ASL) con un Perceptrón Multicapa
## Clasificación de las letras A, E y S

**Curso:** TLY1102 — Machine Learning Avanzado
**Dataset:** Sign Language MNIST (Kaggle)
**Variante del grupo:** clasificación de 3 clases (A, E, S)

---

## a. Introducción

El lenguaje de señas es la principal vía de comunicación de la comunidad sorda. Un sistema capaz de
reconocer automáticamente las letras del alfabeto dactilológico (finger-spelling) puede facilitar la
comunicación entre personas sordas y oyentes que no conocen el lenguaje de señas, sin necesidad de un
intérprete presente.

Sin embargo, no todas las letras son igual de fáciles de reconocer: existen configuraciones de mano muy
similares entre sí (puños cerrados, dedos parcialmente doblados) que generan alta confusión incluso para
observadores humanos. Nuestro grupo decidió enfocar el proyecto específicamente en tres de las letras más
difíciles de distinguir: **A**, **E** y **S**, ya que aportar valor en el caso más desafiante es más
significativo que resolver el caso fácil.

**Objetivo del modelo:** entrenar un Perceptrón Multicapa (MLP) que, a partir de una imagen de una mano
haciendo una seña, prediga si corresponde a la letra A, E o S.

### Sobre el dataset (verificación de las características reales)

> **Nota importante:** la descripción inicial del dataset en la propuesta del proyecto indicaba imágenes
> RGB, 42 clases y ~20.000 imágenes. Al inspeccionar los archivos reales del dataset "Sign Language MNIST"
> de Kaggle, se confirmó que las características reales son distintas. Se documentan a continuación las
> características verificadas, que son las que se usan en el resto del notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

import random

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# Rutas relativas (el notebook vive en notebooks/, los datos en data/)
DATA_DIR = "../data"
MODELS_DIR = "../models"
IMAGES_DIR = "../images"

# Letras que abordará este proyecto (variante del grupo)
LETRAS_OBJETIVO = ["A", "E", "S"]

## b. Comprensión y exploración del conjunto de datos

**Sign Language MNIST** contiene imágenes de manos representando letras del alfabeto dactilológico
americano (ASL), en el mismo formato que el clásico MNIST de dígitos:

- Imágenes de **28×28 píxeles en escala de grises** (1 canal), ya aplanadas en 784 columnas (`pixel1`...`pixel784`).
- **24 clases** (letras A–Y, excluyendo **J** y **Z** porque en ASL real se representan con movimiento,
  no con una postura estática de la mano).
- **27.455 imágenes de entrenamiento** y **7.172 imágenes de test**, entregadas como dos archivos CSV.
- Cada fila = 1 imagen. Primera columna `label` (entero), luego 784 columnas de intensidad de píxel (0–255).

El label es la posición de la letra en el alfabeto (A=0, B=1, ..., I=8, K=10, ..., Y=24), saltándose el 9 (J).

In [ ]:
df_train_full = pd.read_csv(f"{DATA_DIR}/sign_mnist_train.csv")
df_test_full = pd.read_csv(f"{DATA_DIR}/sign_mnist_test.csv")

print("Train:", df_train_full.shape)
print("Test :", df_test_full.shape)
df_train_full.head()

In [ ]:
# Mapeo label numérico -> letra (el dataset salta la letra J = índice 9; Z tampoco está presente)
label_to_letter = {i: chr(65 + i) for i in range(25) if i != 9}
letter_to_label = {v: k for k, v in label_to_letter.items()}

print("Clases presentes en el dataset completo:", sorted(df_train_full['label'].unique()))
print("N° de clases:", df_train_full['label'].nunique())
print()
print("Letras de interés y su label original:")
for letra in LETRAS_OBJETIVO:
    print(f"  {letra} -> label {letter_to_label[letra]}")

In [ ]:
# Distribución de clases en el dataset COMPLETO (contexto, antes de filtrar)
plt.figure(figsize=(12, 4))
conteo = df_train_full['label'].map(label_to_letter).value_counts().sort_index()
conteo.plot(kind='bar', color='steelblue')
plt.title("Distribución de clases en el dataset completo (train)")
plt.xlabel("Letra")
plt.ylabel("N° de imágenes")
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{IMAGES_DIR}/distribucion_clases_completa.png", dpi=120)
plt.show()

In [ ]:
# Visualización de ejemplos de cada letra objetivo
fig, axes = plt.subplots(1, len(LETRAS_OBJETIVO), figsize=(4 * len(LETRAS_OBJETIVO), 4))

for ax, letra in zip(axes, LETRAS_OBJETIVO):
    label_num = letter_to_label[letra]
    fila = df_train_full[df_train_full['label'] == label_num].iloc[0]
    img = fila.drop('label').values.reshape(28, 28).astype('uint8')
    ax.imshow(img, cmap='gray')
    ax.set_title(f"Letra: {letra}")
    ax.axis('off')

plt.suptitle("Ejemplo de cada letra objetivo")
plt.tight_layout()
plt.savefig(f"{IMAGES_DIR}/ejemplos_letras_objetivo.png", dpi=120)
plt.show()

In [ ]:
# Varias muestras por letra, para observar variabilidad intra-clase
fig, axes = plt.subplots(len(LETRAS_OBJETIVO), 6, figsize=(14, 3 * len(LETRAS_OBJETIVO)))

for i, letra in enumerate(LETRAS_OBJETIVO):
    label_num = letter_to_label[letra]
    muestras = df_train_full[df_train_full['label'] == label_num].sample(6, random_state=SEED)
    for j, (_, fila) in enumerate(muestras.iterrows()):
        img = fila.drop('label').values.reshape(28, 28).astype('uint8')
        axes[i, j].imshow(img, cmap='gray')
        axes[i, j].axis('off')
        if j == 0:
            axes[i, j].set_ylabel(letra, fontsize=14)
    axes[i, 0].set_title(f"Letra {letra} — 6 muestras distintas", loc='left', fontsize=11)

plt.tight_layout()
plt.savefig(f"{IMAGES_DIR}/variabilidad_intraclase.png", dpi=120)
plt.show()

**Observaciones del EDA:**

- Las tres letras elegidas (A, E, S) corresponden a configuraciones de **puño cerrado** con variaciones
  sutiles en la posición del pulgar y los dedos, lo que las hace visualmente muy parecidas entre sí —
  justamente la razón por la que el grupo las eligió como el caso más desafiante.
- Se observa variabilidad intra-clase por diferencias de iluminación, ángulo de la mano y tono de piel,
  lo que puede dificultar aún más la separación entre A/E/S respecto a letras con formas más distintivas
  (como el dataset original explora con las 24 clases).
- Al tratarse de imágenes en escala de grises de baja resolución (28×28), gran parte de la información de
  textura y profundidad de la mano se pierde, lo que anticipa una de las limitaciones que se discutirá en
  la sección de análisis de errores.

## c. Preparación y preprocesamiento de los datos

Pasos aplicados:

1. **Filtrado de clases**: nos quedamos solo con las filas cuyo `label` corresponde a A, E o S.
2. **Remapeo de etiquetas**: los labels originales (0, 4, 18) se convierten en índices consecutivos
   (0, 1, 2) requeridos por la capa de salida `softmax` de 3 neuronas.
3. **Normalización**: los valores de píxel (0–255) se escalan a [0, 1] dividiendo por 255, lo que
   estabiliza y acelera el entrenamiento (evita gradientes con magnitudes muy distintas entre variables).
4. **Separación de variables**: `X` (784 columnas de píxeles) e `y` (etiqueta remapeada).
5. **Split de validación**: el archivo de test se deja intacto para la evaluación final; desde el train
   se separa un 20% adicional, estratificado por clase, como conjunto de validación.
6. **One-hot encoding**: `y` se convierte a vector one-hot de 3 posiciones para usar `categorical_crossentropy`.

In [ ]:
def filtrar_y_preparar(df, letter_to_label, letras_objetivo):
    labels_objetivo = [letter_to_label[l] for l in letras_objetivo]
    df_filtrado = df[df['label'].isin(labels_objetivo)].copy()

    # Remapeo a índices consecutivos 0..n_clases-1, en orden alfabético
    mapa_nuevo = {orig: i for i, orig in enumerate(sorted(labels_objetivo))}
    df_filtrado['label_nuevo'] = df_filtrado['label'].map(mapa_nuevo)

    X = df_filtrado.drop(columns=['label', 'label_nuevo']).values.astype('float32') / 255.0
    y = df_filtrado['label_nuevo'].values

    return X, y, mapa_nuevo

X_train_full, y_train_full, mapa_labels = filtrar_y_preparar(df_train_full, letter_to_label, LETRAS_OBJETIVO)
X_test, y_test, _ = filtrar_y_preparar(df_test_full, letter_to_label, LETRAS_OBJETIVO)

# índice nuevo -> letra, para reportes legibles
idx_to_letter = {i: label_to_letter[orig] for orig, i in mapa_labels.items()}
print("Mapeo índice de clase -> letra:", idx_to_letter)

print("X_train_full:", X_train_full.shape, " y_train_full:", y_train_full.shape)
print("X_test:", X_test.shape, " y_test:", y_test.shape)

In [ ]:
# Split de validación estratificado (80% train / 20% val)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.20,
    stratify=y_train_full,
    random_state=SEED
)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

# Verificar que el split mantuvo el balance de clases
for nombre, y_split in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    valores, cuentas = np.unique(y_split, return_counts=True)
    print(nombre, dict(zip([idx_to_letter[v] for v in valores], cuentas)))

In [ ]:
N_CLASES = len(LETRAS_OBJETIVO)

y_train_cat = to_categorical(y_train, num_classes=N_CLASES)
y_val_cat = to_categorical(y_val, num_classes=N_CLASES)
y_test_cat = to_categorical(y_test, num_classes=N_CLASES)

print("y_train_cat shape:", y_train_cat.shape)

## d. Diseño del Perceptrón Multicapa

**Decisiones de arquitectura y su justificación:**

| Decisión | Elección | Justificación |
|---|---|---|
| Capas ocultas | 2 | El problema es de baja complejidad relativa (3 clases muy parecidas, imágenes pequeñas de 28×28), por lo que una red moderadamente profunda es suficiente. Más capas aumentan el riesgo de sobreajuste dado el tamaño acotado del subconjunto de datos (3 de 24 clases). |
| Neuronas por capa | 128 → 64 | Se reduce progresivamente la dimensionalidad (embudo), siguiendo el patrón visto en clase (`Dense(16)`→`Dense(8)`). Con 784 entradas, 128 y 64 dan capacidad suficiente para separar patrones sin sobredimensionar el modelo. |
| Activación capas ocultas | ReLU | Evita el desvanecimiento del gradiente que sí afecta a sigmoide/tanh, y es el estándar en la práctica (visto en la Unidad 1.1 y 1.3). |
| Capa de salida | `Dense(3, softmax)` | Problema de clasificación multiclase (3 clases mutuamente excluyentes): softmax entrega una distribución de probabilidad que suma 1 sobre A/E/S. |
| Función de pérdida | `categorical_crossentropy` | Es la función de pérdida coherente con salida softmax + etiquetas one-hot, como se explicó en Unidad 1.3 (Backpropagation) y se usó en el notebook de MNIST (Unidad 1.4). |
| Optimizador | Adam | Combina momentum y escalamiento adaptativo del learning rate; es la opción por defecto recomendada en el material del curso (Unidad 1.3, PPT Optimizadores) por ser robusto en la mayoría de los casos frente a SGD puro. |
| Learning rate | 0.001 (default de Adam) | Punto de partida estándar; se puede ajustar si las curvas muestran oscilación (muy alto) o convergencia muy lenta (muy bajo). |
| Métrica de monitoreo | accuracy | Métrica intuitiva para seguimiento durante el entrenamiento; se complementa con precision/recall/F1 en la evaluación final porque accuracy sola puede ser engañosa si hay algún desbalance entre clases. |
| Épocas | 40 (con Early Stopping) | Suficientes para observar convergencia; Early Stopping evita sobreentrenar si la pérdida de validación deja de mejorar. |
| Batch size | 64 | Valor intermedio estándar; balancea velocidad de entrenamiento y estabilidad del gradiente. |

In [ ]:
def construir_modelo(input_dim=784, n_clases=3):
    modelo = Sequential([
        Dense(128, activation='relu', input_shape=(input_dim,)),
        Dense(64, activation='relu'),
        Dense(n_clases, activation='softmax')
    ])
    modelo.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return modelo

modelo = construir_modelo(input_dim=X_train.shape[1], n_clases=N_CLASES)
modelo.summary()

## e. Entrenamiento del modelo

Se entrena registrando la evolución de `loss` y `accuracy` tanto en entrenamiento como en validación,
usando `EarlyStopping` para detener el entrenamiento si la pérdida de validación deja de mejorar
(evitando sobreajuste innecesario y tiempo de cómputo extra).

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

historia = modelo.fit(
    X_train, y_train_cat,
    validation_data=(X_val, y_val_cat),
    epochs=40,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(historia.history['loss'], label='Train loss')
axes[0].plot(historia.history['val_loss'], label='Val loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (categorical crossentropy)')
axes[0].set_title('Evolución de la función de pérdida')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(historia.history['accuracy'], label='Train accuracy')
axes[1].plot(historia.history['val_accuracy'], label='Val accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Evolución de la accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{IMAGES_DIR}/curvas_entrenamiento.png", dpi=120)
plt.show()

**Interpretación de las curvas (completar tras ejecutar):**

- ¿La pérdida de entrenamiento y validación bajan juntas o se separan? Una separación creciente
  (train mejora, val empeora o se estanca) es señal de **sobreajuste**.
- ¿La accuracy converge cerca de un valor alto rápidamente, o mejora muy lento? Convergencia muy lenta
  o estancada en valores bajos sugiere **underfitting** (modelo insuficiente, learning rate muy bajo,
  o preprocesamiento inadecuado).
- ¿En qué época se activó el Early Stopping (si se activó)? Indica el punto donde el modelo dejó de
  generalizar mejor.

## f. Evaluación del desempeño

Se evalúa el modelo en el conjunto de **test** (nunca visto durante entrenamiento ni validación),
reportando las métricas mínimas exigidas: Accuracy, Precision, Recall, F1-Score y Matriz de Confusión.

Al ser un problema **multiclase**, Precision/Recall/F1 se calculan con promedio `macro` (cada clase
pesa igual, independiente de su frecuencia) para no ocultar el mal desempeño en una clase minoritaria.

In [ ]:
probs_test = modelo.predict(X_test)
y_pred_test = np.argmax(probs_test, axis=1)

acc = accuracy_score(y_test, y_pred_test)
prec = precision_score(y_test, y_pred_test, average='macro')
rec = recall_score(y_test, y_pred_test, average='macro')
f1 = f1_score(y_test, y_pred_test, average='macro')

print(f"Accuracy : {acc:.4f}")
print(f"Precision (macro): {prec:.4f}")
print(f"Recall    (macro): {rec:.4f}")
print(f"F1-score  (macro): {f1:.4f}")
print()

nombres_clases = [idx_to_letter[i] for i in range(N_CLASES)]
print(classification_report(y_test, y_pred_test, target_names=nombres_clases))

In [ ]:
cm = confusion_matrix(y_test, y_pred_test)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=nombres_clases, yticklabels=nombres_clases)
plt.xlabel('Predicción')
plt.ylabel('Etiqueta real')
plt.title('Matriz de Confusión — Test set')
plt.tight_layout()
plt.savefig(f"{IMAGES_DIR}/matriz_confusion.png", dpi=120)
plt.show()

**Interpretación de las métricas (completar tras ejecutar):**

- **Accuracy** indica el porcentaje total de aciertos, pero por sí sola no dice *cuáles* clases se
  confunden entre sí.
- **Precision** por clase responde: de todas las veces que el modelo predijo esa letra, ¿cuántas
  eran realmente esa letra? Relevante si el "costo" de un falso positivo importa.
- **Recall** por clase responde: de todas las imágenes reales de esa letra, ¿cuántas detectó el modelo?
  Relevante si el "costo" de no detectar la letra correcta importa (p. ej. en un sistema de traducción
  en tiempo real, perder letras rompe la comunicación).
- **F1-score** balancea precision y recall; útil para comparar clases de forma resumida.
- La **matriz de confusión** es la herramienta clave aquí: permite ver exactamente si, por ejemplo,
  el modelo confunde A con S (formas de puño similares) más que con E.

## g. Análisis de resultados y errores

Se realiza una auditoría del modelo revisando ejemplos concretos: casos correctamente clasificados y
casos donde el modelo se equivocó, buscando patrones visuales que expliquen los errores.

In [ ]:
aciertos_idx = np.where(y_pred_test == y_test)[0]
errores_idx = np.where(y_pred_test != y_test)[0]

print(f"Total test: {len(y_test)}")
print(f"Correctamente clasificados: {len(aciertos_idx)} ({len(aciertos_idx)/len(y_test)*100:.1f}%)")
print(f"Mal clasificados: {len(errores_idx)} ({len(errores_idx)/len(y_test)*100:.1f}%)")

In [ ]:
# Ejemplos correctamente clasificados
n_mostrar = min(8, len(aciertos_idx))
muestra_ok = np.random.choice(aciertos_idx, n_mostrar, replace=False)

fig, axes = plt.subplots(1, n_mostrar, figsize=(2 * n_mostrar, 2.5))
for ax, idx in zip(axes, muestra_ok):
    img = (X_test[idx] * 255).reshape(28, 28).astype('uint8')
    ax.imshow(img, cmap='gray')
    ax.set_title(f"Real: {idx_to_letter[y_test[idx]]}\nPred: {idx_to_letter[y_pred_test[idx]]}", fontsize=9)
    ax.axis('off')
plt.suptitle("Ejemplos CORRECTAMENTE clasificados")
plt.tight_layout()
plt.savefig(f"{IMAGES_DIR}/ejemplos_correctos.png", dpi=120)
plt.show()

In [ ]:
# Ejemplos mal clasificados
n_mostrar_err = min(8, len(errores_idx))
if n_mostrar_err > 0:
    muestra_err = np.random.choice(errores_idx, n_mostrar_err, replace=False)

    fig, axes = plt.subplots(1, n_mostrar_err, figsize=(2 * n_mostrar_err, 2.5))
    if n_mostrar_err == 1:
        axes = [axes]
    for ax, idx in zip(axes, muestra_err):
        img = (X_test[idx] * 255).reshape(28, 28).astype('uint8')
        ax.imshow(img, cmap='gray')
        ax.set_title(f"Real: {idx_to_letter[y_test[idx]]}\nPred: {idx_to_letter[y_pred_test[idx]]}",
                     fontsize=9, color='red')
        ax.axis('off')
    plt.suptitle("Ejemplos MAL clasificados")
    plt.tight_layout()
    plt.savefig(f"{IMAGES_DIR}/ejemplos_errores.png", dpi=120)
    plt.show()
else:
    print("El modelo no cometió errores en el test set.")

In [ ]:
# ¿Qué pares de clases se confunden más? (a partir de la matriz de confusión, sin la diagonal)
cm_sin_diagonal = cm.copy()
np.fill_diagonal(cm_sin_diagonal, 0)

print("Pares de clases más confundidos (real -> predicho : cantidad):")
pares = []
for i in range(N_CLASES):
    for j in range(N_CLASES):
        if i != j and cm_sin_diagonal[i, j] > 0:
            pares.append((idx_to_letter[i], idx_to_letter[j], cm_sin_diagonal[i, j]))

pares.sort(key=lambda x: -x[2])
for real, pred, cant in pares:
    print(f"  {real} -> {pred} : {cant} veces")

**Posibles causas de las predicciones erróneas (a completar/discutir con los resultados obtenidos):**

- **Similitud geométrica real entre A, E y S**: las tres corresponden a variantes de puño cerrado; la
  diferencia principal está en la posición exacta del pulgar, un detalle fino que se pierde en la
  baja resolución de 28×28 píxeles.
- **Pérdida de información 3D**: al ser imágenes 2D en escala de grises, se pierde la profundidad de
  la mano, que en la seña real es clave para distinguir estas letras.
- **El MLP no tiene ninguna noción de estructura espacial**: al aplanar la imagen a un vector de 784
  valores, el modelo pierde por completo la relación de vecindad entre píxeles (qué píxel está al lado
  de cuál), a diferencia de una red convolucional (CNN) que sí explota esa estructura 2D mediante filtros
  locales. Esta es la limitación central de usar un MLP para clasificación de imágenes.
- **Variabilidad de iluminación, ángulo y tono de piel** dentro de cada clase (vista en el EDA), que un
  modelo simple como el MLP tiene más dificultad para "ignorar" que una CNN con invariancia espacial.

**Limitaciones del uso de un MLP para clasificación de imágenes (discusión general):**

1. No aprovecha la estructura espacial 2D de la imagen (pierde información de vecindad de píxeles al aplanar).
2. La cantidad de parámetros crece muy rápido con el tamaño de la imagen (aquí 784 entradas ya generan
   decenas de miles de parámetros solo en la primera capa), lo que limita su escalabilidad a imágenes
   más grandes o a color (RGB).
3. No es invariante a traslaciones, rotaciones ni escala — una misma seña desplazada unos píxeles es
   tratada como un patrón de entrada completamente distinto.
4. En consecuencia, arquitecturas más adecuadas para este problema serían Redes Neuronales Convolucionales
   (CNN), que sí explotan la estructura local y son invariantes a pequeñas transformaciones espaciales.

## h. Conclusiones

*(Completar esta sección después de ejecutar el notebook con los resultados reales obtenidos.)*

- **Desempeño alcanzado**: reportar aquí accuracy/precision/recall/F1 finales en test, y si el desempeño
  fue adecuado dado el objetivo (distinguir A/E/S, letras notoriamente similares).
- **Principales decisiones de diseño adoptadas**: arquitectura 128→64→3, ReLU + softmax, Adam,
  categorical_crossentropy, Early Stopping — resumir cuáles tuvieron mayor impacto observado.
- **Fortalezas**: pipeline reproducible, EDA que fundamenta la elección de clases difíciles, evaluación
  con métricas múltiples (no solo accuracy) y análisis de errores basado en evidencia visual.
- **Limitaciones observadas**: las discutidas en la sección g (pérdida de estructura espacial al usar MLP,
  confusión entre letras geométricamente similares).
- **Mejoras futuras posibles**:
  - Migrar a una arquitectura **CNN** (Conv2D + MaxPooling), que debería mejorar sustancialmente el
    desempeño al aprovechar la estructura 2D de la imagen.
  - **Data augmentation** (pequeñas rotaciones/traslaciones) para mejorar la robustez del modelo.
  - Ampliar el análisis a las 24 clases completas y comparar el desempeño específico en A/E/S dentro
    de ese contexto más amplio.
  - Probar distintos hiperparámetros de forma sistemática (grid search sobre arquitectura, learning
    rate y batch size) en vez de una única configuración fija.

## Guardar el modelo entrenado

In [ ]:
modelo.save(f"{MODELS_DIR}/mlp_asl_aes.keras")
print(f"Modelo guardado en {MODELS_DIR}/mlp_asl_aes.keras")